In [23]:
import numpy as np
import pandas as pd
from pathlib import Path
import pickle

In [24]:
PROJ_ROOT = Path().resolve().parents[0]
DATA_DIR = PROJ_ROOT / "data"
INTERIM_DATA_DIR = DATA_DIR / "interim"

In [25]:
X_test = pd.read_pickle(INTERIM_DATA_DIR / "X_test_processed.pkl")
X_train = pd.read_pickle(INTERIM_DATA_DIR / "X_train_processed.pkl")
y_test = pd.read_pickle(INTERIM_DATA_DIR / "y_test_processed.pkl")
y_train = pd.read_pickle(INTERIM_DATA_DIR / "y_train_processed.pkl")

In [26]:
X_train.head(5)

,housing,duration,month_sin,month_cos,day_sin,day_cos,job_admin,job_blue-collar,job_entrepreneur,job_housemaid,...,job_services,job_student,job_technician,job_unemployed,marital_divorced,marital_married,marital_single,education_primary,education_secondary,education_tertiary
31488,1,2.107981,8.660254e-01,-0.500000,0.201299,0.979530,0,1,0,0,...,0,0,0,0,0,1,0,0,1,0
11946,1,-0.234742,1.224647e-16,-1.000000,-0.790776,-0.612106,0,0,0,0,...,0,0,0,0,0,0,1,0,0,1
14355,1,-0.422535,-5.000000e-01,-0.866025,0.299363,-0.954139,0,0,0,0,...,0,0,1,0,0,1,0,0,0,1
39108,1,1.539906,5.000000e-01,-0.866025,-0.485302,-0.874347,0,1,0,0,...,0,0,0,0,0,0,1,0,1,0
25294,0,0.924883,-5.000000e-01,0.866025,-0.485302,-0.874347,0,1,0,0,...,0,0,0,0,0,1,0,0,1,0


<h3>Sampling

A good guide on sampling: https://www.analyticsvidhya.com/blog/2020/07/10-techniques-to-deal-with-class-imbalance-in-machine-learning/

Ways to overcome class imbalance:
1. Undersampling: Smaller proportion of major class relative to the minor class are sampled leading to a more balanced dataset. Can lead to information loss.
2. Oversampling: Copies of the minority class are made to make the dataset more balanced. Can lead to overfishing.

- Both techniques can be done randomly or via more specialist techniques.
- The imblearn library can be utilised for various sampling techniques.


Here we will carry out various sampling techniques, test how this affects a logistic regression technique, and decide which technique is the most effective.

In [49]:
from imblearn.under_sampling import RandomUnderSampler, TomekLinks, NearMiss
from imblearn.over_sampling import RandomOverSampler, SMOTE
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from collections import Counter
from evaluate_binary_classifier import evaluate_binary_classifier

In [28]:
# 1. Random Under Sampling (US)

r_us = RandomUnderSampler(random_state=13, replacement=True)
x_rus, y_rus = r_us.fit_resample(X_train, y_train)

print('original dataset shape:', Counter(y_train))
print('Resample dataset shape', Counter(y_rus))

# We observe a massive shrinkage in the overall size of the training dataset. Are we losing too much information?

original dataset shape: Counter({0: 24861, 1: 1950})
Resample dataset shape Counter({0: 1950, 1: 1950})


In [29]:
lr_model = LogisticRegression(penalty='l2',C=1.0,solver='lbfgs')
lr_model.fit(x_rus,y_rus)
y_pred = lr_model.predict(X_test)
y_proba = lr_model.predict_proba(X_test)[:,1]

lr_eval = evaluate_binary_classifier(y_test=y_test,y_pred=y_pred,y_proba=y_proba)

# We get a high f1-score for the negative class, but not for the positive class e.e. the one that matters here. I think this may be due to the massive class imbalance
# present in the test set. Remember that we have only carried out sampling on the train set. You shouldn't carry out sampling on the test set as it should remain as close
# to the real-world distribution as possible. Perhaps info has been lost by undersampling...


===== Binary Classification Evaluation =====

Accuracy: 0.8459
Balanced Accuracy: 0.8184
Precision: 0.2932
Recall: 0.7862
F1 Score: 0.4271
Matthews Corrcoef: 0.4175
ROC AUC: 0.9004
Log Loss: 0.4455

Confusion Matrix:
[[9089 1596]
 [ 180  662]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.85      0.91     10685
           1       0.29      0.79      0.43       842

    accuracy                           0.85     11527
   macro avg       0.64      0.82      0.67     11527
weighted avg       0.93      0.85      0.88     11527



c:\Users\cochr\Data Science Projects\lA82lBNDpo9AlAnK\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [30]:
# 2. Random Over Sampling (OS)

r_os = RandomOverSampler(random_state=13)
x_ros, y_ros = r_os.fit_resample(X_train, y_train)

print('original dataset shape:', Counter(y_train))
print('Resample dataset shape', Counter(y_ros))

original dataset shape: Counter({0: 24861, 1: 1950})
Resample dataset shape Counter({0: 24861, 1: 24861})


In [31]:
lr_model = LogisticRegression(penalty='l2',C=1.0,solver='lbfgs')
lr_model.fit(x_ros,y_ros)
y_pred = lr_model.predict(X_test)
y_proba = lr_model.predict_proba(X_test)[:,1]

lr_eval = evaluate_binary_classifier(y_test=y_test,y_pred=y_pred,y_proba=y_proba)

# Here we see no difference in the F1-score. It sits comfortably at 43%


===== Binary Classification Evaluation =====

Accuracy: 0.8455
Balanced Accuracy: 0.8193
Precision: 0.2929
Recall: 0.7886
F1 Score: 0.4271
Matthews Corrcoef: 0.4180
ROC AUC: 0.9007
Log Loss: 0.4420

Confusion Matrix:
[[9082 1603]
 [ 178  664]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.85      0.91     10685
           1       0.29      0.79      0.43       842

    accuracy                           0.85     11527
   macro avg       0.64      0.82      0.67     11527
weighted avg       0.93      0.85      0.88     11527



c:\Users\cochr\Data Science Projects\lA82lBNDpo9AlAnK\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [ ]:
# 3. Tomek links US:

# Close (based on features) pairs of instances which are opposite. The majority class of each pair is removed to
# increase the space between classes. Non-random sampling technique.

tl = TomekLinks(sampling_strategy='majority')
x_tl, y_tl = tl.fit_resample(X_train, y_train)

print('original dataset shape:', Counter(y_train))
print('Resample dataset shape', Counter(y_tl))

# Only 600 rows are removed. There's still a massive class imbalance...

original dataset shape: Counter({0: 24861, 1: 1950})
Resample dataset shape Counter({0: 24219, 1: 1950})


In [62]:
lr_model = LogisticRegression(penalty='l2',C=1.0,solver='lbfgs')
lr_model.fit(x_tl,y_tl)
y_pred = lr_model.predict(X_test)
y_proba = lr_model.predict_proba(X_test)[:,1]

lr_eval = evaluate_binary_classifier(y_test=y_test,y_pred=y_pred,y_proba=y_proba)


===== Binary Classification Evaluation =====

Accuracy: 0.9292
Balanced Accuracy: 0.6314
Precision: 0.5289
Recall: 0.2827
F1 Score: 0.3684
Matthews Corrcoef: 0.3531
ROC AUC: 0.8990
Log Loss: 0.1961

Confusion Matrix:
[[10473   212]
 [  604   238]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.98      0.96     10685
           1       0.53      0.28      0.37       842

    accuracy                           0.93     11527
   macro avg       0.74      0.63      0.67     11527
weighted avg       0.92      0.93      0.92     11527



c:\Users\cochr\Data Science Projects\lA82lBNDpo9AlAnK\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [46]:
# 4. Synthetic Minority Oversampling Technique (SMOTE) OS:

# Choose a minority class, find KNN, then place a synthetic point between the point and its chosen neighbour. Repeat until
# the data is balanced.

smote = SMOTE(k_neighbors=5)
x_sm, y_sm = smote.fit_resample(X_train, y_train)

print('original dataset shape:', Counter(y_train))
print('Resample dataset shape', Counter(y_sm))

original dataset shape: Counter({0: 24861, 1: 1950})
Resample dataset shape Counter({0: 24861, 1: 24861})


In [47]:
lr_model = LogisticRegression(penalty='l2',C=1.0,solver='lbfgs')
lr_model.fit(x_sm,y_sm)
y_pred = lr_model.predict(X_test)
y_proba = lr_model.predict_proba(X_test)[:,1]

lr_eval = evaluate_binary_classifier(y_test=y_test,y_pred=y_pred,y_proba=y_proba)

# Slight increase in f1-score to 46%


===== Binary Classification Evaluation =====

Accuracy: 0.8738
Balanced Accuracy: 0.8083
Precision: 0.3339
Recall: 0.7316
F1 Score: 0.4585
Matthews Corrcoef: 0.4376
ROC AUC: 0.8924
Log Loss: 0.3680

Confusion Matrix:
[[9456 1229]
 [ 226  616]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.88      0.93     10685
           1       0.33      0.73      0.46       842

    accuracy                           0.87     11527
   macro avg       0.66      0.81      0.69     11527
weighted avg       0.93      0.87      0.89     11527



c:\Users\cochr\Data Science Projects\lA82lBNDpo9AlAnK\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [65]:
x_sm.head()

,housing,duration,month_sin,month_cos,day_sin,day_cos,job_admin,job_blue-collar,job_entrepreneur,job_housemaid,...,job_services,job_student,job_technician,job_unemployed,marital_divorced,marital_married,marital_single,education_primary,education_secondary,education_tertiary
0,1,2.107981,8.660254e-01,-0.500000,0.201299,0.979530,0,1,0,0,...,0,0,0,0,0,1,0,0,1,0
1,1,-0.234742,1.224647e-16,-1.000000,-0.790776,-0.612106,0,0,0,0,...,0,0,0,0,0,0,1,0,0,1
2,1,-0.422535,-5.000000e-01,-0.866025,0.299363,-0.954139,0,0,0,0,...,0,0,1,0,0,1,0,0,0,1
3,1,1.539906,5.000000e-01,-0.866025,-0.485302,-0.874347,0,1,0,0,...,0,0,0,0,0,0,1,0,1,0
4,0,0.924883,-5.000000e-01,0.866025,-0.485302,-0.874347,0,1,0,0,...,0,0,0,0,0,1,0,0,1,0


In [63]:
# 5. NearMiss US:

# Instead of randomly removing majority class samples, removal is done based on their distance to minority class samples.

nm = NearMiss(n_neighbors=3,sampling_strategy=1)

x_nm, y_nm = nm.fit_resample(X_train, y_train)

print('Original dataset shape:', Counter(y_train))
print('Resample dataset shape:', Counter(y_nm))

Original dataset shape: Counter({0: 24861, 1: 1950})
Resample dataset shape: Counter({0: 1950, 1: 1950})


In [64]:
lr_model = LogisticRegression(penalty='l2',C=1.0,solver='lbfgs')
lr_model.fit(x_nm,y_nm)
y_pred = lr_model.predict(X_test)
y_proba = lr_model.predict_proba(X_test)[:,1]

lr_eval = evaluate_binary_classifier(y_test=y_test,y_pred=y_pred,y_proba=y_proba)


===== Binary Classification Evaluation =====

Accuracy: 0.6805
Balanced Accuracy: 0.6991
Precision: 0.1497
Recall: 0.7209
F1 Score: 0.2479
Matthews Corrcoef: 0.2170
ROC AUC: 0.7834
Log Loss: 0.7576

Confusion Matrix:
[[7237 3448]
 [ 235  607]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.68      0.80     10685
           1       0.15      0.72      0.25       842

    accuracy                           0.68     11527
   macro avg       0.56      0.70      0.52     11527
weighted avg       0.91      0.68      0.76     11527



c:\Users\cochr\Data Science Projects\lA82lBNDpo9AlAnK\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Based on these techniques, it seems like SMOTE is the most effecive technique. It deals with the imbalance whilst preserving information due to its non-random nature.

In [68]:
# Sample and save the data to pkl files:

smote = SMOTE(k_neighbors=15)
x_sm, y_sm = smote.fit_resample(X_train, y_train)

print('original dataset shape:', Counter(y_train))
print('Resample dataset shape', Counter(y_sm))

lr_model = LogisticRegression(penalty='l2',C=1.0,solver='lbfgs')
lr_model.fit(x_sm,y_sm)
y_pred = lr_model.predict(X_test)
y_proba = lr_model.predict_proba(X_test)[:,1]

lr_eval = evaluate_binary_classifier(y_test=y_test,y_pred=y_pred,y_proba=y_proba)

original dataset shape: Counter({0: 24861, 1: 1950})
Resample dataset shape Counter({0: 24861, 1: 24861})

===== Binary Classification Evaluation =====

Accuracy: 0.8913
Balanced Accuracy: 0.7866
Precision: 0.3656
Recall: 0.6639
F1 Score: 0.4715
Matthews Corrcoef: 0.4397
ROC AUC: 0.8874
Log Loss: 0.3239

Confusion Matrix:
[[9715  970]
 [ 283  559]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.91      0.94     10685
           1       0.37      0.66      0.47       842

    accuracy                           0.89     11527
   macro avg       0.67      0.79      0.71     11527
weighted avg       0.93      0.89      0.91     11527



c:\Users\cochr\Data Science Projects\lA82lBNDpo9AlAnK\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [69]:
x_sm.to_pickle(INTERIM_DATA_DIR / "X_train_sampled.pkl")
y_sm.to_pickle(INTERIM_DATA_DIR / "y_train_sampled.pkl")

Now onto modelling...